In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

## BRCA1

### Findlay 2018

Comeback to this. It is hg37 :(

In [ ]:
b18f = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Findlay2018.xlsx", 
    sheet_name="Sheet1", 
    read_options={"skip_rows": 2},
    has_header=False
)

# Step 1: extract first row
new_header = b18f.row(0)  # returns a tuple

# Step 2: assign as new header and drop first row
b18f = b18f.slice(1).rename({old: new for old, new in zip(b18f.columns, new_header)})
# b18f.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Findlay2018.parquet')
b18f

In [ ]:
b18f_vcf = b18f.rename({
    'position (hg19)':'pos',
    'reference':'ref',
}).with_columns(
    ('chr' + pl.col('chromosome')).alias('chrom'),
).with_columns(
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id'),
).select(
    ['chrom', 'pos', 'id', 'ref', 'alt']
).with_columns(
    pl.lit('.').alias('QUAL'),
    pl.lit('.').alias('FILTER'),
    pl.lit('.').alias('INFO'),
)

b18f_vcf

In [ ]:
vcf_header = f"""
##fileformat=VCFv4.2
##source=BRCA1_SGE_Findlay2018
##INFO=<ID=NS,Number=1,Type=Integer,Description="Number of Samples With Data">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
"""

output_file = "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Findlay2018_hg37.vcf"

with open(output_file, "w") as f:
    # Write the static header
    f.write(vcf_header)
    
    # Append the DataFrame content as tab-separated values without its own header
    b18f_vcf.write_csv(f, separator='\t', include_header=False)

### Findlay 2018 - hg38

In [ ]:
b18f = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Findlay2018.xlsx", 
    sheet_name="Sheet1", 
    read_options={"skip_rows": 2},
    has_header=False
)

new_header = b18f.row(0)

b18f = b18f.slice(1).rename({old: new for old, new in zip(b18f.columns, new_header)}).rename({
    'position (hg19)':'pos',
    'reference':'ref',
    'function.score.mean': 'score'
}).with_columns(
    ('chr' + pl.col('chromosome')).alias('chrom'),
).with_columns(
    pl.col('pos').cast(pl.Int64),
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id'),
)

b18f

In [ ]:
b18f_file = '/s/project/solve_rd/deeprvat/UKBB_data/BRCA1_Findlay2018_hg38.vcf'

b18f_map = pl.read_csv(b18f_file, separator='\t', comment_prefix='##').rename({
    '#CHROM':'chrom',
    'POS':'pos',
    'ID':'id',
    'REF':'ref',
    'ALT':'alt',
}).drop(['QUAL', 'FILTER', 'INFO'])
b18f_map

In [ ]:
b18f_df = b18f_map.join(b18f[['id', 'score', 'aa_pos', 'aa_ref', 'aa_alt']], on=['id'], how='inner').with_columns(
    pl.col('score').cast(pl.Float64),
    pl.lit('BRCA1').alias('gene_name'),
    pl.lit('ENSG00000012048').alias('region'),
    pl.lit('SGE').alias('assay_name'),
    pl.lit('BRCA1_SGE').alias('assay_description'),
    pl.lit('BRCA1_SGE_Findlay2018').alias('source'),
).with_columns(
    (pl.col('aa_ref') + pl.col('aa_pos').cast(pl.Utf8) + pl.col('aa_alt')).alias('mutant'),
    (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id')
).drop(['aa_ref', 'aa_pos', 'aa_alt'])

b18f_df

In [ ]:
b18f_df.columns

In [ ]:
b18f_df.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_SGE_Findlay2018.parquet')

### Dace 2025

In [ ]:
b25d_common = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025.xlsx", 
    sheet_name="ST3", 
    read_options={"skip_rows": 3},
    has_header=False
)

# Step 1: extract first row
new_header = b25d_common.row(0)  # returns a tuple

# Step 2: assign as new header and drop first row
b25d_common = b25d_common.slice(1).rename({old: new for old, new in zip(b25d_common.columns, new_header)})
b25d_common

In [ ]:
b25d_df = b25d_common.rename({
    'hg38': 'pos',
    'Ref': 'ref',
    'Alt': 'alt',
    'HAP1_function_score_mean': 'SGE_HAP1',
    'final_function_score_ut': 'SGE_HMEC_ut',
    'final_function_score_olaparib': 'SGE_HMEC_olaparib'
}).with_columns(
    pl.col('pos').cast(pl.Int64),
    pl.lit('chr17').alias('chrom'),
    pl.lit('BRCA1').alias('gene_name'),
    pl.lit('ENSG00000012048').alias('region'),
    pl.lit('BRCA1_SGE_Dace2025').alias('source'),
    (pl.col('oAA') + pl.col('protPos').cast(pl.Utf8) + pl.col('nAA')).alias('mutant'),
).select(
    ['chrom', 'pos', 'ref', 'alt', 'mutant', 'gene_name', 'region', 'source', 'SGE_HAP1', 'SGE_HMEC_ut', 'SGE_HMEC_olaparib']
).unpivot(
    index=['chrom', 'pos', 'ref', 'alt', 'mutant', 'gene_name', 'region', 'source'],
    on=['SGE_HAP1', 'SGE_HMEC_ut', 'SGE_HMEC_olaparib'],
    variable_name='assay_name',
    value_name='score'
).with_columns(
    ('BRCA1_' + pl.col('assay_name')).alias('assay_description'),
).with_columns(
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id'),
    (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id')
).filter(pl.col('score')!="NA").with_columns(
    pl.col('score').cast(pl.Float64)
)

b25d_df

In [ ]:
b25d_df.pivot(
    index=['id', 'region'],
    columns='assay_id',
    values='score'
)

In [ ]:
b25d_df.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_SGE_Dace2025.parquet')

In [ ]:
b25d_hap1 = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca1/BRCA1_Dace2025.xlsx", 
    sheet_name="ST1", 
    read_options={"skip_rows": 3},
    has_header=False
)

# Step 1: extract first row
new_header = b25d_hap1.row(0)  # returns a tuple

# Step 2: assign as new header and drop first row
b25d_hap1 = b25d_hap1.slice(1).rename({old: new for old, new in zip(b25d_hap1.columns, new_header)})
b25d_hap1

## BRCA2

### Sahu 2025

In [ ]:
b25s = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_Sahu2025.xlsx", 
    sheet_name="Sheet 1"
)
b25s

In [ ]:
pattern = r"(\d+)([A-Z]+)>([A-Z]+)"

b25s_df = b25s.with_columns(
    pl.lit('BRCA2_Sahu2025').alias('source'),
    pl.lit('BRCA2_Sahu2025_SGE_EScell').alias('assay_description'),
    pl.lit('SGE_EScell').alias('assay_name'),
    pl.lit('BRCA2').alias('gene_name'),
    pl.lit('ENSG00000139618').alias('region'),
    pl.lit('chr13').alias('chrom'),
    pl.col('g.nom').str.split('.').list.get(2).str.extract_groups(pattern).alias("parsed_variant")
).unnest(
    # Step 2: Expand the struct fields into new columns
    "parsed_variant"
).rename(
    # Step 3: Rename the new columns to what you want.
    { "1": "pos", "2": "ref", "3": "alt", "AA.change": "mutant", "Function.score": "score"}
).with_columns(
    pl.col('pos').cast(pl.Int64),
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id'),
    (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id'),
).select(
    ['source', 'assay_description', 'assay_id', 'assay_name', 'gene_name', 'region', 'id', 'mutant', 'score']
).drop_nulls()

b25s_df

In [ ]:
b25s_df.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_SGE_Sahu2025.parquet')

### Huang 2025

In [ ]:
b25h = pl.read_excel(
    "/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_Huang2025.xlsx", 
    sheet_name="Table S3",
    read_options={"skip_rows": 1},
    has_header=False
)

new_header = b25h.row(0)  

b25h = b25h.slice(1).rename({old: new for old, new in zip(b25h.columns, new_header)})
b25h

In [ ]:
b25h_df = b25h.with_columns(
    pl.lit('BRCA2_Huang2025').alias('source'),
    pl.lit('BRCA2_Huang2025_SGE_HAP1').alias('assay_description'),
    pl.lit('SGE_HAP1').alias('assay_name'),
    pl.lit('BRCA2').alias('gene_name'),
    pl.lit('ENSG00000139618').alias('region'),
    pl.lit('chr13').alias('chrom'),
).rename({
    "GRCh38Location": "pos", 
    "REF": "ref", 
    "ALT": "alt", 
    "Amino acid change (p.)": "mutant", 
    "Model based functional score": "score",
    "Posterior probability of pathogenicity": "post_prob_patho"
}).with_columns(
    pl.col('pos').cast(pl.Int64),
    pl.col('score').cast(pl.Float64),
    pl.col('post_prob_patho').cast(pl.Float64),
    (pl.col('chrom') + ':' + pl.col('pos').cast(pl.Utf8) + ':' + pl.col('ref') + ':' + pl.col('alt')).alias('id'),
    (pl.col('assay_name') + '_' + pl.col('source') + '_' + pl.col('assay_description').str.replace(" ", "_")).alias('assay_id'),
).select(
    ['source', 'assay_description', 'assay_id', 'assay_name', 'gene_name', 'region', 'id', 'mutant', 'score']
).drop_nulls()

b25h_df

In [ ]:
b25h_df['mutant']

In [ ]:
b25h_df.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/brca2/BRCA2_SGE_Huang2025.parquet')

In [ ]:
b25h_df[['id', 'score']].join(b25s_df[['id', 'score']], on='id', how='inner').rename({'score': 'score_huang', 'score_right': 'score_sahu'})